In [1]:
# location of parquet file on farm
# /group/ctbrowngrp5/sra-metagenomes/20241128-metadata.parquet

In [2]:
import polars as pl

In [3]:
df = pl.read_parquet('/group/ctbrowngrp5/sra-metagenomes/20241128-metadata.parquet')

In [4]:
# read list of bioProjects
with open('bioProjects_AMD.txt') as f:
    my_bioprojects = [line.strip() for line in f if line.strip()]
my_bioprojects

['PRJNA1125351',
 'PRJNA554371',
 'PRJNA666025',
 'PRJNA699454',
 'PRJNA723739',
 'PRJNA801253',
 'PRJNA954676',
 'PRJDB19179',
 'PRJNA646106']

In [5]:

# makes subtable of just accessions for desired bioproject list. 
# OUTCOME: 103 samples
df2 = (
    df
    .filter(pl.col('bioproject').is_in(my_bioprojects))
    .select(['acc', 'sample_name', 'bioproject'])
)
df2


acc,sample_name,bioproject
str,str,str
"""SRR13197465""","""DX1-meta""","""PRJNA666025"""
"""SRR13201267""","""HSP2-meta""","""PRJNA666025"""
"""SRR13213328""","""YF4-meta""","""PRJNA666025"""
"""SRR13205816""","""LL3-meta""","""PRJNA666025"""
"""SRR13213330""","""YF3-meta""","""PRJNA666025"""
…,…,…
"""SRR13190158""","""MAS1-meta""","""PRJNA666025"""
"""SRR13304735""","""WH5-meta""","""PRJNA666025"""
"""SRR13188941""","""FK1-meta""","""PRJNA666025"""


In [6]:
# saved in sra_accs_AMD.txt
sra_accs = df2['acc'].to_list()

with open('sra_accessions.txt', 'w') as f:
    for acc in sra_accs:
        f.write(acc + '\n')
print(f"Got {len(sra_accs)} accessions")
print("Wrote sra_accessions.txt")
print(f"test metagenome accs: {sra_accs[:3]}")  # the first 3

Got 103 accessions
Wrote sra_accessions.txt
test metagenome accs: ['SRR13197465', 'SRR13201267', 'SRR13213328']


In [7]:
# confirm reference is here still 
#       '!ls ' is a one line bash command
!ls /home/ctbrown/scratch3/2026-gtdb-dl/gtdb-cds-rs226.species.singleton.sig.zip

!ls /group/ctbrowngrp5/wort/wort-sra/sigs/SRR13197465.sig

/home/ctbrown/scratch3/2026-gtdb-dl/gtdb-cds-rs226.species.singleton.sig.zip
/group/ctbrowngrp5/wort/wort-sra/sigs/SRR13197465.sig


In [8]:
from pathlib import Path

sigs_dir = Path("/group/ctbrowngrp5/wort/wort-sra/sigs")
accessions = Path("/home/hehouts/AMD/sra_accessions.txt").read_text().splitlines()

present = [a for a in accessions if (sigs_dir / f"{a}.sig").exists()]
missing = [a for a in accessions if a not in present]

if present:
    print(f"found {len(present)} of {len(accessions)} files")
#    for a in present:
#        print(f"  {a}")
#    if missing:
#        print("no hits for:")
#        for a in missing:
#            print(f"  {a}")
else:
    print("struck out :/")

found 103 of 103 files


In [11]:
!sourmash scripts manysearch --help


== This is sourmash version 4.9.4. ==
== Please cite Irber et. al (2024), doi:10.21105/joss.06830. ==

usage:  manysearch [-h] [-q] [-d] -o OUTPUT [-t THRESHOLD] [-k KSIZE]
                   [-s SCALED] [-m {DNA,protein,dayhoff,hp,skipm1n3,skipm2n3}]
                   [-c CORES] [-P] [-N] [--ignore-abundance] [-A]
                   query_paths against_paths

search many metagenomes for contained genomes

positional arguments:
  query_paths           input file of sketches
  against_paths         input file of sketches

options:
  -h, --help            show this help message and exit
  -q, --quiet           suppress non-error output
  -d, --debug           provide debugging output
  -o OUTPUT, --output OUTPUT
                        CSV output file for matches
  -t THRESHOLD, --threshold THRESHOLD
                        containment threshold for reporting matches (default:
                        0.01)
  -k KSIZE, --ksize KSIZE
                        k-mer size at which to select 

In [ ]:
'SRR13197465', 'SRR13201267', 'SRR13213328'

In [ ]:
#"
# positional arguments:
#     query_paths           input file of sketches
#     against_paths         input file of sketches

#"

In [ ]:
%%bash
# Search Test (one) metagenome, SRR13197465
# Path to metagenome /group/ctbrowngrp5/wort/wort-sra/sigs/SRR13197465.sig

sourmash scripts manysearch \
  -k 21 \
  --scaled 1000 \
  --threshold=0 \
  /home/ctbrown/scratch3/2026-gtdb-dl/gtdb-cds-rs226.species.singleton.sig.zip \
  /group/ctbrowngrp5/wort/wort-sra/sigs/SRR13197465.sig \
  -o output.csv


== This is sourmash version 4.9.4. ==
== Please cite Irber et. al (2024), doi:10.21105/joss.06830. ==

=> sourmash_plugin_branchwater 0.9.15; cite Irber et al., doi: 10.1101/2022.11.02.514947

ksize: 21 / scaled: 1000 / moltype: DNA / threshold: 0.0
searching all sketches in '/home/ctbrown/scratch3/2026-gtdb-dl/gtdb-cds-rs226.species.singleton.sig.zip' against '/group/ctbrowngrp5/wort/wort-sra/sigs/SRR13197465.sig' using 1 threads
selection scaled: Some(1000)
Reading query(s) from: '/home/ctbrown/scratch3/2026-gtdb-dl/gtdb-cds-rs226.species.singleton.sig.zip'
Loaded 143390 query signature(s)
Reading search(s) from: '/group/ctbrowngrp5/wort/wort-sra/sigs/SRR13197465.sig'
Loaded 1 search signature(s)
DONE. Processed 1 search sigs
...manysearch is done! results in 'output.csv'



query             p_genome avg_abund   p_metag   metagenome name
--------          -------- ---------   -------   ---------------
GCA_000008085 s__    0.3%     2.0       0.0%     SRR13197465
GCA_000008885 s__    1.6%     2.8       0.0%     SRR13197465
GCA_000017645 s__    0.6%     1.5       0.0%     SRR13197465
GCA_000018565 s__    0.2%     1.2       0.0%     SRR13197465
GCA_000023725 s__    0.1%     2.5       0.0%     SRR13197465
GCA_000024525 s__    0.1%     1.0       0.0%     SRR13197465
GCA_000026125 s__    0.4%     1.2       0.0%     SRR13197465
GCA_000091165 s__    0.2%     1.0       0.0%     SRR13197465
GCA_000143165 s__    0.1%     1.0       0.0%     SRR13197465
GCA_000145985 s__    0.2%     1.0       0.0%     SRR13197465
GCA_000152145 s__    0.1%     1.0       0.0%     SRR13197465
GCA_000153145 s__    0.4%     1.3       0.0%     SRR13197465
GCA_000153385 s__    0.1%     1.5       0.0%     SRR13197465
GCA_000155695 s__    0.2%     1.1       0.0%     SRR13197465
GCA_000155715 s

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



05 s__    0.6%     1.0       0.0%     SRR13197465
GCA_009702465 s__    0.3%     3.0       0.0%     SRR13197465
GCA_009702555 s__    0.3%     1.0       0.0%     SRR13197465
GCA_009702755 s__    0.2%     1.0       0.0%     SRR13197465
GCA_009702775 s__    0.1%     1.0       0.0%     SRR13197465
GCA_009703065 s__    0.2%     1.0       0.0%     SRR13197465
GCA_009703445 s__    0.2%     1.0       0.0%     SRR13197465
GCA_009703455 s__    0.1%     1.0       0.0%     SRR13197465
GCA_009703465 s__    0.3%     1.5       0.0%     SRR13197465
GCA_009703585 s__    0.2%     1.0       0.0%     SRR13197465
GCA_009703905 s__    0.1%     1.0       0.0%     SRR13197465
GCA_009704065 s__    0.1%     1.0       0.0%     SRR13197465
GCA_009704335 s__    0.2%     8.0       0.0%     SRR13197465
GCA_009704345 s__    0.2%     2.0       0.0%     SRR13197465
GCA_009704425 s__    0.4%     1.0       0.0%     SRR13197465
GCA_009704445 s__    0.2%     1.0       0.0%     SRR13197465
GCA_009704715 s__    0.1%     1.0  

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



0.0%     SRR13197465
GCA_017393775 s__    0.3%     1.0       0.0%     SRR13197465
GCA_017394205 s__    0.1%     1.0       0.0%     SRR13197465
GCA_017394445 s__    0.2%     4.5       0.0%     SRR13197465
GCA_017394515 s__    0.2%     1.5       0.0%     SRR13197465
GCA_017394565 s__    0.3%     1.0       0.0%     SRR13197465
GCA_017394705 s__    0.2%     1.0       0.0%     SRR13197465
GCA_017394775 s__    0.3%     1.5       0.0%     SRR13197465
GCA_017394805 s__    0.3%     4.0       0.0%     SRR13197465
GCA_017394835 s__    0.4%     2.2       0.0%     SRR13197465
GCA_017394865 s__    0.3%     1.7       0.0%     SRR13197465
GCA_017395045 s__    0.2%    39.5       0.0%     SRR13197465
GCA_017395065 s__    0.4%     1.4       0.0%     SRR13197465
GCA_017395215 s__    0.4%     1.0       0.0%     SRR13197465
GCA_017395245 s__    0.2%     1.7       0.0%     SRR13197465
GCA_017395265 s__    0.3%     1.0       0.0%     SRR13197465
GCA_017395425 s__    0.1%     4.0       0.0%     SRR13197465
GCA

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



0.2%     1.0       0.0%     SRR13197465
GCA_021730905 s__    0.3%     1.2       0.0%     SRR13197465
GCA_021730935 s__    0.3%     3.8       0.0%     SRR13197465
GCA_021730945 s__    0.6%     1.2       0.0%     SRR13197465
GCA_021730985 s__    0.5%     1.4       0.0%     SRR13197465
GCA_021731005 s__    0.2%     3.0       0.0%     SRR13197465
GCA_021731025 s__    0.2%     3.5       0.0%     SRR13197465
GCA_021731045 s__    0.2%     2.5       0.0%     SRR13197465
GCA_021731085 s__    0.4%     5.8       0.0%     SRR13197465
GCA_021731095 s__    0.4%     1.0       0.0%     SRR13197465
GCA_021731145 s__    0.3%     1.0       0.0%     SRR13197465
GCA_021731205 s__    1.1%     1.0       0.0%     SRR13197465
GCA_021731215 s__    0.3%     1.0       0.0%     SRR13197465
GCA_021731245 s__    0.3%     6.0       0.0%     SRR13197465
GCA_021731285 s__    0.6%     2.4       0.0%     SRR13197465
GCA_021731325 s__    0.2%     7.0       0.0%     SRR13197465
GCA_021731365 s__    0.5%     9.7       0.0% 

In [13]:
!head -n 5 output.csv

In [ ]:
%%bash
# Search Test (one) metagenome, SRR13197465
# Path to metagenome /group/ctbrowngrp5/wort/wort-sra/sigs/SRR13197465.sig

sourmash scripts manysearch \
  -k 21 \
  --scaled 1000 \
  --threshold=0 \
  /home/ctbrown/scratch3/2026-gtdb-dl/gtdb-cds-rs226.species.singleton.sig.zip \
  /group/ctbrowngrp5/wort/wort-sra/sigs/SRR13197465.sig \
  -o output.csv

In [ ]:
scripts: error: argument subcmd: invalid choice: 'manysearch' (choose from )
---------------------------------------------------------------------------
CalledProcessError                        Traceback (most recent call last)
Cell In[8], line 1
----> 1 get_ipython().run_cell_magic('bash', '', '# Search Test (one) metagenome, SRR13197465\n# Path to metagenome /group/ctbrowngrp5/wort/wort-sra/sigs/SRR13197465.sig\n\nsourmash scripts manysearch \\\n  -k 21 \\\n  --scaled 1000 \\\n  --threshold=0 \\\n  /home/ctbrown/scratch3/2026-gtdb-dl/gtdb-cds-rs226.species.singleton.sig.zip \\\n  /group/ctbrowngrp5/wort/wort-sra/sigs/SRR13197465.sig \\\n  -o output.csv\n')

CalledProcessError: Command 'b'# Search Test (one) metagenome, SRR13197465\n# Path to metagenome /group/ctbrowngrp5/wort/wort-sra/sigs/SRR13197465.sig\n\nsourmash scripts manysearch \\\n  -k 21 \\\n  --scaled 1000 \\\n  --threshold=0 \\\n  /home/ctbrown/scratch3/2026-gtdb-dl/gtdb-cds-rs226.species.singleton.sig.zip \\\n  /group/ctbrowngrp5/wort/wort-sra/sigs/SRR13197465.sig \\\n  -o output.csv\n'' returned non-zero exit status 2.

In [10]:
!pwd

/home/hehouts/AMD


In [ ]:
# Path to singletons
/home/ctbrown/scratch3/2026-gtdb-dl/gtdb-cds-rs226.species.singleton.sig.zip

In [5]:
df2 = df.select(['acc', 'organism', 'mbases'])
df2

acc,organism,mbases
str,str,i64
"""SRR28523869""","""human metagenome""",258
"""SRR19901137""","""Streptococcus suis""",40
"""SRR24962475""","""Escherichia coli""",88
"""SRR29161330""","""biofilm metagenome""",3
"""SRR26668739""","""bovine gut metagenome""",129
…,…,…
"""ERR2709388""","""human gut metagenome""",411
"""SRR089825""","""Neurospora crassa""",530
"""SRR14870612""","""Frankiales bacterium""",15


In [16]:
df10 = df2.filter(pl.col("organism") == "human gut metagenome").drop_nulls().sort(by='mbases', descending=True).head(10000)
df10.write_csv('/tmp/yanshan.csv')
df10

acc,organism,mbases
str,str,i64
"""SRR10238604""","""human gut metagenome""",256947
"""ERR7803603""","""human gut metagenome""",209223
"""SRR12137097""","""human gut metagenome""",173611
"""SRR21422847""","""human gut metagenome""",173281
"""ERR8158273""","""human gut metagenome""",151153
…,…,…
"""ERR4097240""","""human gut metagenome""",11832
"""SRR9333187""","""human gut metagenome""",11831
"""SRR8849213""","""human gut metagenome""",11831


In [6]:
df3 = df.select('organism')

In [11]:
df3['organism'].value_counts().sort('count', descending=True)

NameError: name 'df3' is not defined

In [8]:
start_keys = df3['organism'].value_counts().sort('count', descending=True)['organism'].to_list()
d = {}
for k in start_keys:
    d[k] = k

d['metagenome'] = 'foo'
d['gut metagenome'] = 'foo'
d['human metagenome'] = 'useless'
d['feces metagenome'] = 'foo'
d['bacterium'] = 'useless'
d['unidentified'] = 'foo'
d['oral metagenome'] = 'useless'
d['skin metagenome'] = 'useless'
d['vaginal metagenome'] = 'useless'

d['Severe acute respiratory syndrome coronavirus 2'] = 'useless'

d['human gut metagenome'] = 'ignore'

d['Homo sapiens'] = 'ignore'
d['Mus musculus'] = 'mouse'


d['human skin metageome'] = 'ignore'
d['human oral metagenome'] = 'ignore'
d['human skin metagenome'] = 'ignore'

d['human vaginal metagenome'] = 'ignore'
d['human nasopharyngeal metagenome'] = 'ignore'
d['human blood metagenome'] = 'ignore'
d['human feces metagenome'] = 'ignore'
d['human lung metagenome'] = 'ignore'


d['mouse gut metagenome'] = 'mouse'
d['mouse metagenome'] = 'mouse'

d['respiratory tract metagenome'] = 'ignore'
#d['pig gut metagenome'] = 'ignore'
d['viral metagenome'] = 'ignore'


In [9]:
replace_df = pl.DataFrame(dict(organism=d.keys(), org2=d.values()))
replace_df

organism,org2
str,str
"""human gut metagenome""","""ignore"""
"""metagenome""","""foo"""
"""soil metagenome""","""soil metagenome"""
"""gut metagenome""","""foo"""
"""human metagenome""","""useless"""
…,…
"""Lactobacillaceae bacterium UBA…","""Lactobacillaceae bacterium UBA…"
"""Massilia sp. UBA2117""","""Massilia sp. UBA2117"""
"""Halieaceae bacterium UBA3479""","""Halieaceae bacterium UBA3479"""


In [10]:
df3 = df2.join(replace_df, on='organism', how='left')
df3

acc,organism,org2
str,str,str
"""SRR28523869""","""human metagenome""","""useless"""
"""SRR19901137""","""Streptococcus suis""","""Streptococcus suis"""
"""SRR24962475""","""Escherichia coli""","""Escherichia coli"""
"""SRR29161330""","""biofilm metagenome""","""biofilm metagenome"""
"""SRR26668739""","""bovine gut metagenome""","""bovine gut metagenome"""
…,…,…
"""ERR2709388""","""human gut metagenome""","""ignore"""
"""SRR089825""","""Neurospora crassa""","""Neurospora crassa"""
"""SRR14870612""","""Frankiales bacterium""","""Frankiales bacterium"""


In [11]:
with pl.Config(tbl_rows=-1):
    print(df3['org2'].value_counts().sort('count', descending=True).filter(
        (pl.col("count") > 1000))
    )

shape: (79, 2)
┌─────────────────────────────────┬────────┐
│ org2                            ┆ count  │
│ ---                             ┆ ---    │
│ str                             ┆ u32    │
╞═════════════════════════════════╪════════╡
│ ignore                          ┆ 378142 │
│ foo                             ┆ 193927 │
│ useless                         ┆ 86786  │
│ soil metagenome                 ┆ 72561  │
│ mouse                           ┆ 54056  │
│ marine metagenome               ┆ 17840  │
│ freshwater metagenome           ┆ 15245  │
│ sediment metagenome             ┆ 14349  │
│ indoor metagenome               ┆ 12833  │
│ wastewater metagenome           ┆ 12634  │
│ pig gut metagenome              ┆ 11341  │
│ rhizosphere metagenome          ┆ 11193  │
│ food metagenome                 ┆ 6846   │
│ plant metagenome                ┆ 6436   │
│ bovine gut metagenome           ┆ 6320   │
│ sheep gut metagenome            ┆ 6284   │
│ synthetic metagenome            ┆ 5795

In [12]:
biglist_sum = df3['org2'].value_counts().sort('count', descending=True).filter(
    pl.col("count") > 3000
)['count'].sum()

In [13]:
print(biglist_sum / len(df3))

0.832115479909512


In [14]:
for k in df3['organism'].unique():
    for i in 'feline', 'canine', 'felis catus':
        if i in k.lower():
            print(k)



feline metagenome
Feline calicivirus
canine metagenome
Feline leukemia virus
canine gut metagenome
Felis catus
Canine respiratory coronavirus
Canine astrovirus


In [15]:
df3

acc,organism,org2
str,str,str
"""SRR28523869""","""human metagenome""","""useless"""
"""SRR19901137""","""Streptococcus suis""","""Streptococcus suis"""
"""SRR24962475""","""Escherichia coli""","""Escherichia coli"""
"""SRR29161330""","""biofilm metagenome""","""biofilm metagenome"""
"""SRR26668739""","""bovine gut metagenome""","""bovine gut metagenome"""
…,…,…
"""ERR2709388""","""human gut metagenome""","""ignore"""
"""SRR089825""","""Neurospora crassa""","""Neurospora crassa"""
"""SRR14870612""","""Frankiales bacterium""","""Frankiales bacterium"""


In [16]:
dogs = ['canine metagenome', 'canine gut metagenome']
cats = ['Felis catus', 'feline metagenome']
both = dogs + cats

xx_df = df3.filter(pl.col('organism').is_in(both))
print(len(xx_df))
print(xx_df.filter(pl.col('organism').is_in(cats)))
xx_df['organism'].value_counts()


1259
shape: (307, 3)
┌─────────────┬───────────────────┬───────────────────┐
│ acc         ┆ organism          ┆ org2              │
│ ---         ┆ ---               ┆ ---               │
│ str         ┆ str               ┆ str               │
╞═════════════╪═══════════════════╪═══════════════════╡
│ SRR17156348 ┆ Felis catus       ┆ Felis catus       │
│ SRR24332709 ┆ feline metagenome ┆ feline metagenome │
│ SRR17156364 ┆ Felis catus       ┆ Felis catus       │
│ ERR8529888  ┆ feline metagenome ┆ feline metagenome │
│ ERR878264   ┆ Felis catus       ┆ Felis catus       │
│ …           ┆ …                 ┆ …                 │
│ ERR8529881  ┆ feline metagenome ┆ feline metagenome │
│ ERR878303   ┆ Felis catus       ┆ Felis catus       │
│ SRR24332707 ┆ feline metagenome ┆ feline metagenome │
│ ERR8577720  ┆ feline metagenome ┆ feline metagenome │
│ SRR24332701 ┆ feline metagenome ┆ feline metagenome │
└─────────────┴───────────────────┴───────────────────┘


organism,count
str,u32
"""Felis catus""",130
"""canine metagenome""",784
"""canine gut metagenome""",168
"""feline metagenome""",177


In [17]:
cat_df = xx_df.filter(pl.col('organism').is_in(cats))
accs = cat_df['acc'].to_list()
open("/tmp/cat-list.txt", "w").write("\n".join(accs))

3399

In [18]:
cat_df = xx_df.filter(pl.col('organism').is_in(dogs))
accs = cat_df['acc'].to_list()
open("/tmp/dog-list.txt", "w").write("\n".join(accs))

11241

In [20]:
from collections import defaultdict
sums = defaultdict(int)

xxx = df3['org2'].value_counts().sort('count', descending=True).iter_rows(named=True)
for row in xxx:
    for word in 'bovine', 'sheep', 'pig', 'Bos taurus', 'goat', 'mouse':
        total = 0
        if word in row['org2']:
            sums[word] += row["count"]

for k, v in sorted(sums.items(), key=lambda x: x[1]):
    print(k, v)

goat 58
Bos taurus 423
sheep 6345
bovine 9260
pig 12805
mouse 54192
